# Chapter 7: Statistical Analysis of Factor Effects

**Learning objectives:**
- Run a full DOE benchmark against an evaluator and collect results
- Analyze factor effects via logistic regression (analysis of deviance)
- Interpret G-squared statistics, pseudo-R², odds ratios and Wilson CIs
- Detect interaction effects between factors
- Identify top failure modes and per-category patterns
- Generate HTML dashboard reports

---
## Preamble

Chapter 6 showed how the geometric judge produces continuous verdicts for individual answers. This chapter closes the loop: we run **all 640 DOE questions** through an evaluator, collect the verdicts and use **logistic regression** to determine which presentation factors (clarity, entity aliasing, reasoning cue) significantly affect accuracy.

Since we do not have a real LLM agent in this tutorial environment, we construct a **simulated evaluator** that returns the correct GMS answer most of the time but injects controlled, factor-dependent failures using a seeded random generator. The failure rates are transparent and fully reproducible — the reader can see exactly what patterns are injected.

Key modules:
- `harness.doe.benchmark` — `DOEGMSBenchmark`, `DOEHarnessConfig`
- `graphdoe.analysis` — `DOEAnalyzer`
- `demo.mock_agent` — `create_mock_evaluator` (for reference)

In [ ]:
# === Preamble (all notebooks) ===
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.doe import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


In [ ]:
from importlib.resources import files
# Load the Chapter 4 store and set up the benchmark
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"\nStore: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## The Simulated Evaluator

A real agent evaluation would call an LLM via `bench.run(evaluator=fn)`. Since we do not have API access here, we build a **simulated evaluator** that:

1. Looks up each question's ground truth from the GMS answer functions
2. Returns the correct answer most of the time
3. Injects failures with probability that depends on (a) question category and (b) DOE factor assignments

This lets us demonstrate the full analysis pipeline with realistic, interpretable factor effects.

In [4]:
# Step 1: Generate questions so we can build the evaluator
doe_questions = bench.generate_questions()
print(f"Generated {len(doe_questions)} DOE questions across "
      f"{len(set(dq.question.category for dq in doe_questions))} categories")

  Base questions: 160 (20 generators, max 10/cat)

  DOE expansion: 160 base x 4 runs = 640 questions
  Factors: ['clarity', 'entity_aliasing', 'reasoning_cue']
  [BackgroundTrain] Starting: 1 entities, 0 relations, 25 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 4.6s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
Generated 640 DOE questions across 16 categories


In [5]:
# Step 2: Define failure rates
#
# Category base failure rates (probability of incorrect answer)
CATEGORY_FAIL = {
    "exact_retrieval": 0.02,  "phase_comparison": 0.02,
    "link_prediction": 0.05, "plausibility": 0.05,
    "workflow_compliance": 0.10, "governance": 0.10,
    "counting": 0.12,        "cross_reference": 0.12,
    "tension": 0.15,         "recovery_test": 0.18,
    "holonomy": 0.22,        "multi_hop_transport": 0.28,
    "multi_hop": 0.35,       "boundary": 0.38,
    "robustness": 0.42,      "adversarial": 0.48,
}

# Factor effects: (factor_name, level) -> additive change in failure probability
FACTOR_EFFECTS = {
    ("clarity", "Misleading"): 0.15,
    ("clarity", "Ambiguous"): 0.05,
    ("clarity", "Clear"): 0.0,
    ("entity_aliasing", "Synonym"): 0.08,
    ("entity_aliasing", "Abbreviated"): 0.04,
    ("entity_aliasing", "Exact"): 0.0,
    ("reasoning_cue", "No_Cue"): 0.03,
    ("reasoning_cue", "Direct"): 0.0,
    ("reasoning_cue", "Step_By_Step"): -0.02,
}

# Print expected accuracy per category (at baseline factor levels)
print(f"{'Category':<25s} {'Base Fail%':>10s} {'Base Acc%':>10s}")
print("-" * 48)
for cat in sorted(CATEGORY_FAIL, key=CATEGORY_FAIL.get):
    pf = CATEGORY_FAIL[cat]
    print(f"{cat:<25s} {pf:>9.0%} {1-pf:>9.0%}")

overall = 1 - np.mean(list(CATEGORY_FAIL.values()))
print(f"\nExpected baseline accuracy (no factor effects): {overall:.1%}")

Category                  Base Fail%  Base Acc%
------------------------------------------------
exact_retrieval                  2%       98%
phase_comparison                 2%       98%
link_prediction                  5%       95%
plausibility                     5%       95%
workflow_compliance             10%       90%
governance                      10%       90%
counting                        12%       88%
cross_reference                 12%       88%
tension                         15%       85%
recovery_test                   18%       82%
holonomy                        22%       78%
multi_hop_transport             28%       72%
multi_hop                       35%       65%
boundary                        38%       62%
robustness                      42%       58%
adversarial                     48%       52%

Expected baseline accuracy (no factor effects): 81.0%


In [6]:
# Step 3: Create the simulated evaluator
#
# Since bench.run() iterates through doe_questions in order, we use a
# sequential counter to match each evaluator call to its DOE question.
# This avoids text-matching issues where multiple DOE runs share the
# same llm_prompt text.

call_idx = [0]  # mutable counter (closure-compatible)
rng = np.random.RandomState(42)

def simulated_evaluator(question: str, context: str) -> str:
    """Evaluator that returns correct answers with factor-dependent failures."""
    idx = call_idx[0]
    call_idx[0] += 1

    if idx >= len(doe_questions):
        return "I cannot determine the answer"

    dq = doe_questions[idx]
    q = dq.question
    ground_truth = str(q.ground_truth)
    category = q.category
    factors = dq.factor_assignment

    # Compute failure probability: base rate + factor effects
    p_fail = CATEGORY_FAIL.get(category, 0.10)
    for (fname, level), delta in FACTOR_EFFECTS.items():
        if factors.get(fname) == level:
            p_fail += delta
    p_fail = np.clip(p_fail, 0.0, 0.95)

    # Deterministic failure decision (seeded RNG)
    if rng.random() < p_fail:
        return "I cannot determine the answer from the available data"
    return ground_truth

simulated_evaluator.__repr__ = lambda: "SimulatedEvaluator(seed=42)"
print("Simulated evaluator created (seed=42, sequential index)")

Simulated evaluator created (seed=42, sequential index)


---

## Lab 7.1 — Run the Full Benchmark

The `DOEGMSBenchmark.run()` method executes the complete evaluation pipeline:

1. For each of the 640 DOE questions: extract context, call evaluator, parse answer, score, geometric judge
2. Return a `DOEBenchmarkResult` with per-question results and factor assignments

With a real agent, replace `simulated_evaluator` with `create_mock_evaluator()` or an LLM-backed callable.

In [7]:
# Lab 7.1: Run the full benchmark
result = bench.run(evaluator=simulated_evaluator)

  [BackgroundTrain] Starting: 3 entities, 0 relations, 10 triples


  [BackgroundTrain] Done: Trained: 3 entities, 0 relations, 50 steps in 4.6s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved

  Evaluating 640 DOE questions (context_mode=full)...
  [1/640] cross_reference_1309_run000... 

FAIL geo=0.20 (0.0s)
  [BackgroundTrain] Starting: 2 entities, 0 relations, 15 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 4.7s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [2/640] cross_reference_228_run000... 

FAIL geo=0.36 (0.0s)
  [BackgroundTrain] Starting: 1 entities, 0 relations, 15 triples


...

PASS geo=0.70 (0.0s)
  [BackgroundTrain] Starting: 505 entities, 0 relations, 15 triples


  [BackgroundTrain] Error: The expanded size of the tensor (4967) must match the existing size (4961) at non-singleton dimension 0.  Target sizes: [4967, 256].  Tensor sizes: [4961, 256]
  [640/640] governance_017_run003... 

PASS geo=0.70 (0.0s)
  [BackgroundTrain] Starting: 507 entities, 0 relations, 15 triples


  [BackgroundTrain] Error: The expanded size of the tensor (4967) must match the existing size (4961) at non-singleton dimension 0.  Target sizes: [4967, 256].  Tensor sizes: [4961, 256]
  [BackgroundTrain] Starting: 510 entities, 0 relations, 4 triples


  [BackgroundTrain] Error: The expanded size of the tensor (4967) must match the existing size (4961) at non-singleton dimension 0.  Target sizes: [4967, 256].  Tensor sizes: [4961, 256]

  Result: 371/640 (58.0%)


In [8]:
# Summary of benchmark results
print(f"{'='*60}")
print("  DOE Benchmark Results")
print(f"{'='*60}")
print("  Model:            SimulatedEvaluator(seed=42)")
print(f"  Total questions:  {result.total_questions}")
print(f"  Total correct:    {result.total_correct}")
print(f"  Accuracy:         {result.accuracy:.1%}")
print(f"  Factors:          {result.factors}")

# Category breakdown
cat_total = Counter()
cat_correct = Counter()
for qr in result.questions:
    cat_total[qr.category] += 1
    if qr.correct:
        cat_correct[qr.category] += 1

print(f"\n  {'Category':<25s} {'Correct':>8s} {'Total':>6s} {'Accuracy':>9s}")
print(f"  {'-'*52}")
for cat in sorted(cat_total.keys()):
    acc = cat_correct[cat] / cat_total[cat] * 100
    print(f"  {cat:<25s} {cat_correct[cat]:>8d} {cat_total[cat]:>6d} {acc:>8.1f}%")

  DOE Benchmark Results
  Model:            SimulatedEvaluator(seed=42)
  Total questions:  640
  Total correct:    371
  Accuracy:         58.0%
  Factors:          ['clarity', 'entity_aliasing', 'reasoning_cue']

  Category                   Correct  Total  Accuracy
  ----------------------------------------------------
  adversarial                     12     40     30.0%
  boundary                        24     40     60.0%
  counting                        32     40     80.0%
  cross_reference                  0     40      0.0%
  exact_retrieval                 35     40     87.5%
  governance                      31     40     77.5%
  holonomy                        25     40     62.5%
  link_prediction                 34     40     85.0%
  multi_hop                        0     40      0.0%
  multi_hop_transport             25     40     62.5%
  phase_comparison                34     40     85.0%
  plausibility                    32     40     80.0%
  recovery_test             

The simulated agent achieves 57.3% overall accuracy with wide category variation: exact retrieval (87.5%) and phase comparison (85%) score highest, while adversarial (30%) and robustness (32.5%) are lowest. Two categories — cross reference and multi-hop — score 0% due to complex answer-type parsing requirements that string comparison cannot resolve. This spread provides the variation needed for meaningful factor analysis.

**The question is: which presentation factors drive these failures?**

---

## Logistic Regression: Analysis of Deviance

Because our response variable is binary (correct/incorrect), we use **logistic regression** rather than ANOVA. For each factor, the `DOEAnalyzer` fits:

$$\text{logit}\bigl(P(\text{correct}=1)\bigr) = \beta_0 + \beta_1 X_{\text{factor}}$$

and tests significance via the **likelihood-ratio (deviance) chi-squared** statistic:

$$G^2 = -2\bigl(\ell_{\text{null}} - \ell_{\text{model}}\bigr) \;\sim\; \chi^2(df)$$

where $df$ = number of factor levels minus 1. A significant $G^2$ means the factor explains more variation in accuracy than chance.

In [9]:
# Create the analyzer and run logistic regression
analyzer = bench.analyze(result)

logistic = analyzer.run_logistic()

print(f"{'Factor':<25s} {'G²':>8s} {'p-value':>10s} {'Pseudo-R²':>10s} {'Sig':>5s}")
print("-" * 62)
for _, row in logistic.iterrows():
    sig = "*" if row["significant"] else ""
    g2 = f"{row['deviance']:.2f}" if not np.isnan(row['deviance']) else "NaN"
    pv = f"{row['p_value']:.4f}" if not np.isnan(row['p_value']) else "NaN"
    print(f"{row['factor']:<25s} {g2:>8s} {pv:>10s} "
          f"{row['pseudo_r2']:>10.4f} {sig:>5s}")

  [BackgroundTrain] Starting: 515 entities, 16 relations, 19 triples


  [BackgroundTrain] Error: The expanded size of the tensor (4967) must match the existing size (4961) at non-singleton dimension 0.  Target sizes: [4967, 256].  Tensor sizes: [4961, 256]
  Store saved to doe_tracing_store/_doe_tracing
Factor                          G²    p-value  Pseudo-R²   Sig
--------------------------------------------------------------
entity_aliasing               9.33     0.0094     0.0107     *
clarity                       9.29     0.0096     0.0107     *
reasoning_cue                 9.29     0.0096     0.0107     *


### Interpreting the Results

- **G² (deviance)**: The likelihood-ratio test statistic. Larger values indicate stronger factor effects.
- **p-value**: Probability of observing this deviance under the null hypothesis (no factor effect). Values below 0.05 are marked significant.
- **Pseudo-R²**: McFadden's pseudo-R², measuring the proportion of deviance explained. Even small values (0.01–0.05) can be practically meaningful for binary outcomes.

### Per-Level Accuracy and Odds Ratios

For each factor, we examine the per-level accuracy rates with Wilson confidence intervals and odds ratios.

In [10]:
# Per-level accuracy rates and odds ratios for each factor
for _, row in logistic.iterrows():
    sig = " *" if row["significant"] else ""
    print(f"\n{row['factor']}{sig} (pseudo-R² = {row['pseudo_r2']:.4f}):")
    print(f"  {'Level':<25s} {'Rate':>8s} {'95% CI':>18s} {'OR':>8s} {'N':>6s}")
    print(f"  {'-'*68}")
    for level, info in sorted(row["level_rates"].items(),
                              key=lambda x: x[1]["rate"]):
        ci = f"[{info['ci_lo']:.3f}, {info['ci_hi']:.3f}]"
        or_val = row["odds_ratios"].get(level, 1.0)
        or_str = f"{or_val:.2f}" if or_val != 1.0 else "ref"
        print(f"  {str(level):<25s} {info['rate']:>7.1%} "
              f"{ci:>18s} {or_str:>8s} {info['n']:>6d}")


entity_aliasing * (pseudo-R² = 0.0107):
  Level                         Rate             95% CI       OR      N
  --------------------------------------------------------------------
  Synonym                     54.1%     [0.486, 0.594]     0.94    320
  Abbreviated                 55.6%     [0.479, 0.631]      ref    160
  Exact                       68.1%     [0.606, 0.748]     1.71    160

clarity * (pseudo-R² = 0.0107):
  Level                         Rate             95% CI       OR      N
  --------------------------------------------------------------------
  Misleading                  53.8%     [0.460, 0.613]     0.95    160
  Ambiguous                   55.0%     [0.495, 0.604]      ref    320
  Clear                       68.1%     [0.606, 0.748]     1.75    160

reasoning_cue * (pseudo-R² = 0.0107):
  Level                         Rate             95% CI       OR      N
  --------------------------------------------------------------------
  Step_By_Step                53

The **odds ratio (OR)** measures the relative odds of a correct answer at each level compared to the reference level (first alphabetically). An OR of 2.0 means the odds of success are twice as high; an OR of 0.5 means half as high.

---

## Interaction Effects

Main effects tell us which factors matter individually. **Interaction effects** tell us whether the combination of two factors has an effect beyond their individual contributions. The analyzer compares an additive model ($A + B$) against an interaction model ($A + B + A{:}B$) using a likelihood-ratio test.

In [11]:
# Interaction analysis: test all factor pairs
from itertools import combinations

all_pairs = list(combinations(result.factors, 2))

try:
    interactions = analyzer.run_interaction_logistic(factor_pairs=all_pairs)
    has_interactions = not interactions.empty
except (KeyError, ValueError):
    interactions = None
    has_interactions = False

if not has_interactions:
    print("No significant interactions detected.")
    print("(All factor pairs tested — none showed interaction effects.)")
    print(f"\nPairs tested: {all_pairs}")
elif "interaction_deviance" in interactions.columns:
    print(f"{'Factor 1':<20s} {'Factor 2':<20s} {'G²':>8s} {'p-value':>10s} "
          f"{'Sig':>5s} {'Strength':>10s}")
    print("-" * 78)
    for _, row in interactions.iterrows():
        sig = "*" if row["significant"] else ""
        print(f"{row['factor_1']:<20s} {row['factor_2']:<20s} "
              f"{row['interaction_deviance']:>8.2f} "
              f"{row['interaction_p_value']:>10.4f} {sig:>5s} "
              f"{row['interaction_strength']:>10.4f}")
else:
    print(f"{'Factor 1':<20s} {'Factor 2':<20s} {'Strength':>10s}")
    print("-" * 55)
    for _, row in interactions.iterrows():
        print(f"{row['factor_1']:<20s} {row['factor_2']:<20s} "
              f"{row['interaction_strength']:>10.4f}")

Factor 1             Factor 2                   G²    p-value   Sig   Strength
------------------------------------------------------------------------------
clarity              entity_aliasing         -0.00     1.0000          -0.0000
entity_aliasing      reasoning_cue           -0.01     1.0000          -0.0000
clarity              reasoning_cue           -0.03     1.0000          -0.0000


Since our simulated failure model uses purely additive effects (each factor contributes independently to the failure probability), we expect no significant interactions — and this is exactly what the analysis confirms. In a real agent evaluation, significant interactions would indicate that certain factor combinations are disproportionately harmful.

---

## Failure Analysis

The failure analysis examines which factor levels are associated with the highest failure rates, with Wilson confidence intervals for reliable estimation even at small sample sizes.

In [12]:
# Top failure modes
failures = analyzer.failure_analysis()
top_failures = failures[failures["failure_rate"] > 0].head(10)

print(f"{'Factor':<20s} {'Level':<20s} {'Fail%':>7s} {'95% CI':>18s} {'N':>6s}")
print("-" * 75)
for _, row in top_failures.iterrows():
    ci = f"[{row['ci_lo']:.3f}, {row['ci_hi']:.3f}]"
    print(f"{row['factor']:<20s} {str(row['level']):<20s} "
          f"{row['failure_rate']:>6.1%} {ci:>18s} {row['n_total']:>6d}")

Factor               Level                  Fail%             95% CI      N
---------------------------------------------------------------------------
clarity              Misleading            46.2%     [0.387, 0.540]    160
reasoning_cue        Step_By_Step          46.2%     [0.387, 0.540]    160
entity_aliasing      Synonym               45.9%     [0.406, 0.514]    320
clarity              Ambiguous             45.0%     [0.396, 0.505]    320
reasoning_cue        No_Cue                45.0%     [0.396, 0.505]    320
entity_aliasing      Abbreviated           44.4%     [0.369, 0.521]    160
clarity              Clear                 31.9%     [0.252, 0.394]    160
entity_aliasing      Exact                 31.9%     [0.252, 0.394]    160
reasoning_cue        Direct                31.9%     [0.252, 0.394]    160


The failure analysis highlights which factor levels are most problematic. These are the conditions to prioritize for agent improvement — the factor level with the highest failure rate indicates the presentation style that most degrades performance.

---

## Per-Category Analysis

Different question categories may be sensitive to different factors. The `by_category()` method runs separate logistic regressions per category to reveal category-specific patterns.

In [13]:
# Per-category significant factors
cat_results = analyzer.by_category()

print(f"{'Category':<25s} {'Significant Factors':s}")
print("-" * 65)
for cat in sorted(cat_results.keys()):
    cat_logistic = cat_results[cat]
    sig = cat_logistic[cat_logistic["significant"]]
    if not sig.empty:
        factors_str = ", ".join(
            f"{r['factor']}(R²={r['pseudo_r2']:.2f})"
            for _, r in sig.iterrows()
        )
        print(f"  {cat:<23s} {factors_str}")
    else:
        cat_acc = analyzer.df[analyzer.df["category"] == cat]["correct"].mean()
        if cat_acc == 1.0:
            print(f"  {cat:<23s} (100% — no variation to analyze)")
        elif cat_acc == 0.0:
            print(f"  {cat:<23s} (0% — no variation to analyze)")
        else:
            print(f"  {cat:<23s} (none significant)")

Category                  Significant Factors
-----------------------------------------------------------------
  adversarial             (none significant)
  boundary                (none significant)
  counting                (none significant)
  cross_reference         (0% — no variation to analyze)
  exact_retrieval         (none significant)
  governance              entity_aliasing(R²=0.22), clarity(R²=0.21), reasoning_cue(R²=0.21)
  holonomy                clarity(R²=0.13), reasoning_cue(R²=0.13)
  link_prediction         (none significant)
  multi_hop               (0% — no variation to analyze)
  multi_hop_transport     (none significant)
  phase_comparison        (none significant)
  plausibility            (none significant)
  recovery_test           (none significant)
  robustness              (none significant)
  tension                 clarity(R²=0.12), reasoning_cue(R²=0.12)
  workflow_compliance     (none significant)


Categories with 100% or 0% accuracy show no factor sensitivity — the agent either always succeeds or always fails, regardless of presentation. The interesting categories are those in between, where factor levels create measurable differences in accuracy.

---

## Lab 7.2 — Full Analysis Report and Dashboard

The `print_report()` method consolidates all analyses into a single comprehensive output.

In [14]:
# Full analysis report
analyzer.print_report()


DOE ANALYSIS REPORT
  Total questions: 640
  Overall accuracy: 58.0%
  Factors tested: ['clarity', 'entity_aliasing', 'reasoning_cue']

--- Analysis of Deviance (Logistic Regression) ---
  Factor                          G²    p-value       R²   Sig
  ────────────────────────────────────────────────────────────
  entity_aliasing               9.33     0.0094    0.011     *
  clarity                       9.29     0.0096    0.011     *
  reasoning_cue                 9.29     0.0096    0.011     *

--- Significant Factors (p < 0.05) ---

  entity_aliasing (pseudo-R² = 0.011):
    Level                         Rate             95% CI       OR
    ──────────────────────────────────────────────────────────────
    Synonym                     54.1%     [0.486, 0.594]     0.94
    Abbreviated                 55.6%     [0.479, 0.631]      ref
    Exact                       68.1%     [0.606, 0.748]     1.71

  clarity (pseudo-R² = 0.011):
    Level                         Rate             95


--- Per-Category Significant Factors ---
  governance           entity_aliasing(R²=0.22), clarity(R²=0.21), reasoning_cue(R²=0.21)
  holonomy             clarity(R²=0.13), reasoning_cue(R²=0.13)
  tension              clarity(R²=0.12), reasoning_cue(R²=0.12)



### Generating the HTML Dashboard

For stakeholder communication, the benchmark produces a self-contained HTML dashboard that includes summary cards, category breakdowns, geometric confidence analysis and factor effect tables.

In [15]:
# Generate HTML dashboard
report_path = bench.generate_report(
    result, analyzer=analyzer,
    output_path="monograph/notebooks/ch07_report.html",
)
print(f"Dashboard saved to: {report_path}")
print(f"File size: {os.path.getsize(report_path) / 1024:.1f} KB")

  Report saved to monograph/notebooks/ch07_report.html
  [BackgroundTrain] Starting: 520 entities, 18 relations, 5 triples


  [BackgroundTrain] Error: The expanded size of the tensor (4967) must match the existing size (4961) at non-singleton dimension 0.  Target sizes: [4967, 256].  Tensor sizes: [4961, 256]
Dashboard saved to: monograph/notebooks/ch07_report.html
File size: 402.0 KB


---

## Exercises

### Exercise 7.1

Modify the `FACTOR_EFFECTS` dictionary to add a strong interaction: increase the failure rate by an additional 0.20 when *both* `clarity=Misleading` and `entity_aliasing=Synonym`. Re-run the benchmark and the interaction analysis. Does the interaction become significant?

### Exercise 7.2

Extract the DataFrame from the benchmark result (`result.to_dataframe()`) and compute the mean geometric confidence (`_overall_confidence`) for each factor level. Which factor level has the lowest geometric confidence? Is it the same level that has the lowest accuracy?

### Exercise 7.3

Run `analyzer.by_category()` and identify the three categories with the strongest factor effects (highest pseudo-R²). For each, explain why that category might be especially sensitive to the dominant factor.

In [16]:
# === Exercise 7.1: Your code here ===
# Hint: add a check in simulated_evaluator for the combination
# if info["factors"].get("clarity") == "Misleading" and \
#    info["factors"].get("entity_aliasing") == "Synonym":
#     p_fail += 0.20

In [17]:
# === Exercise 7.2: Your code here ===
# df = result.to_dataframe()
# for factor in result.factors:
#     print(df.groupby(factor)["_overall_confidence"].mean())

In [18]:
# === Exercise 7.3: Your code here ===

---

## Summary

In this chapter we completed the core evaluation pipeline by running 640 DOE questions through a simulated evaluator and analyzing the results statistically:

1. **The benchmark pipeline** (ingest → generate → run → analyze) produces a `DOEBenchmarkResult` with per-question correctness, factor assignments and geometric scores.
2. **Logistic regression** (analysis of deviance) tests whether each presentation factor significantly affects accuracy. The G² statistic, p-value and pseudo-R² quantify the strength of each factor's effect.
3. **Odds ratios** and **Wilson confidence intervals** provide interpretable effect sizes — how much more (or less) likely the agent is to answer correctly at each factor level.
4. **Interaction analysis** tests whether factor combinations have effects beyond their individual contributions.
5. **Failure analysis** identifies the factor levels associated with the highest failure rates.
6. **Per-category analysis** reveals that different question types may be sensitive to different factors.
7. **HTML dashboards** package all results into stakeholder-ready reports.

**Next:** Part III introduces the typed verification pipeline — decomposing agent outputs into verifiable claims (Chapter 8), attaching specialized verifiers (Chapter 9) and producing detailed verdicts with failure codes and severity (Chapter 10).